# Imports

In [6]:
import numpy as np
import math
from typing import Any



In [56]:
l: float #: Stack length [m]
D: float #: Rotor diameter [m]
B_delta_m: float #: Peak air-gap flux density [T]
v: int #: Space-harmonic index
p: int #: Pole pairs
Q_s: int #: Stator slot count
k_fe: float #: Iron fill factor
b_t_I: float #: Tooth width [m]
k_h: float #: Hysteresis loss coeff
a_h: float #: Hysteresis exponent a_h
b_h: float #: Hysteresis exponent b_h
k_e: float #: Eddy-current loss coeff
f: float #: Frequency by harmonic
Kx_Fe: float #: Stacking factor
ro_c: float #: Core density [kg/m^3]
P_2: float #: Output mechanical power [W]
n_delta: float #: Air gap efficiency [-]
T_n: float #: Torque [Nm]



# --------------------------Magnet losses---------------------------------------
N_seg_tan: int # Number of tangential segments [-]
N_seg_ax: int # Number of axial segments [-]
lm_seg: float # Axial length of each segment [m]
bo_marked: float # Total width of the area where the slot opening has an impact on the rotor surface flux density [-]
beta: float  # Parameter for calculation [-]
u: float #Parameter for calculation [-]
alpha_1_v: float #Parameter for calculation [-]


# Input parameters

In [57]:
Do = 0.2                                        #TODO
hs = 27.3e-3                                    #TODO
h1 = 2.391183e-3                                #TODO
h2 = 13.25104e-3                                #TODO
h3 = 11.045623e-3                               #TODO

b_o = 3.5e-3                                    #TODO
h_o = 0.64e-3
tau_s = 0.013                                   #TODO
b_t = 7e-3                                      #TODO
k_h = 0.022418                                  #TODO
a_h = 1.6978                                    #TODO
b_h = -0.232595                                 #TODO
B_t_I = 1.24                                    #TODO
B_t_II_0 = 1.24                                 #TODO
B_t_II_half = 1.43                              #TODO
B_t_II_h_1 = 1.68                               #TODO
B_t_III = 1.68                                  #TODO
B_ys = 1.08                                     #TODO
ro_c = 7600                                     #TODO
Kx_Fe = 1.7                                     #TODO
Q_s = 27                                        #TODO
k_e = 5.163168e-5                               #TODO
B_delta_m = 0.88                                #TODO
alpha_m = 130                                   #TODO
D = 0.11                                        #TODO
l = 0.12                                        #TODO
p = 3                                           #TODO
k_fe = 0.97                                     #TODO
h_ys = 17.7e-3                                  #TODO




In [58]:
b_t_I = tau_s - b_o
b_t_II_0 = tau_s - b_o
b_t_II_half = (b_t_I + b_t) / 2
b_t_II_h_1 = b_t
b_t_III = b_t





# Core losses

In [ ]:
B_t_I_v = Any
B_t_II_v_0 = Any
B_t_II_v_half = Any
B_t_II_v_h_1 = Any
B_t_III_v = Any
Psi_t_v = Any
B_delta_m_v = Any
Pc_new = 50000
Pc_old = 0
Pct_I = 0
Pct_II = 0
Pct_III = 0
P_cys = 0
P_total = 0
# v_B_t_I = 0
# v_B_t_II_0 = 0
# v_B_t_II_half = 0
# v_B_t_II_h_1 = 0
# v_B_t_III = 0
# v_B_ysv = 0
f = 75

iter = 1
v = 1
n = 2
while (abs(Pc_old - Pc_new)/Pc_new) > 0.01:


    Pc_old = Pc_new
    Pc_new = 0


    B_delta_m_v = (4/math.pi) * (1/v) * B_delta_m * abs(math.sin(v * (alpha_m * math.pi / 180)/2  ))

    Psi_t_v = l * (D/2) * B_delta_m_v * ( 2 / (v*p) ) * abs(math.sin( v * p * math.pi / Q_s ))

    Psi_ys_v = l * (D/2) * B_delta_m_v * (1/ (v * p) )

    B_t_I_v = Psi_t_v / (k_fe * b_t_I * l)

    B_t_II_v_0 = Psi_t_v / (k_fe * b_t_II_0 * l)

    B_t_II_v_half = Psi_t_v / (k_fe * b_t_II_half * l)

    B_t_II_v_h_1 = Psi_t_v / (k_fe * b_t_II_h_1 * l)

    B_t_III_v = Psi_t_v / (k_fe * b_t_III * l)

    B_ysv = Psi_ys_v / (k_fe * h_ys * l)



    Pct_I += Kx_Fe*Q_s * (k_h * f * B_t_I**(a_h + b_h*B_t_I) + k_e * f**2 * (v * B_t_I_v)**2) * ro_c * k_fe * b_t_I * h_o * l
    Pc_new += Pct_I

    Pct_II += Kx_Fe * Q_s * ro_c * k_fe * l * (h1 / 6) * ( (k_h * f * B_t_II_0 **(a_h + b_h*B_t_II_0) + k_e * f**2 * (v * B_t_II_0)**2 ) * b_t_II_0 + \
    4 * (k_h * f * B_t_II_half**(a_h + b_h*B_t_II_half) + k_e * f**2 * (v * B_t_II_v_half)**2 ) * b_t_II_half + \
    (k_h * f * B_t_II_h_1**(a_h + b_h*B_t_II_h_1) + k_e * f**2 * (v * B_t_II_h_1)**2 ) * b_t_II_h_1                                                      )
    Pc_new += Pct_II

    Pct_III += Kx_Fe * Q_s * (k_h * f * B_t_III**(a_h + b_h*B_t_III) + k_e * f**2 * (v * B_t_III_v)**2) * ro_c * k_fe * b_t_III * (h2 + h3) * l
    Pc_new += Pct_III

    P_cys += Kx_Fe * (k_h * f * B_ys**(a_h + b_h*B_ys) + k_e * f**2 * (v * B_ysv)**2) * ro_c * k_fe * (Do**2 - (D + 2 * hs)**2 ) * (math.pi / 4) * l
    Pc_new += P_cys


    P_total = Pct_I + Pct_II + Pct_III + P_cys

    print(f"In iteration {iter}. the difference was {(abs(Pc_old - Pc_new)/Pc_new) * 100} ")


    iter += 1

    n += 1

    v += 2


# Magnet losses